# MODIS MCD19A2 and Himawari-8/9 AOD Merging Workflow (Vietnam)


## **Scientific Workflow:**
1. Load Himawari TIFF as the base grid.
2. Extract MODIS AOD, QA, and Viewing Angles directly from the HDF4 file.
3. Reproject MODIS to perfectly match the Himawari grid.
4. Train a Machine Learning model (Random Forest) to normalize MODIS viewing geometry to Himawari's static perspective, using only the highest quality pixels.
5. Apply an Uncertainty-Weighted Conditional Merge to gap-fill the dataset.





## Step 0: Imports and Configuration
Ensure you have the required libraries installed:
`pip install numpy pandas xarray rioxarray rasterio scikit-learn`

In [ ]:
import os
import numpy as np
import pandas as pd
import xarray as xr
import rioxarray as rxr
import rasterio
from rasterio.enums import Resampling
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import warnings

In [ ]:

import glob
import pandas as pd
from datetime import datetime

In [ ]:
# ==========================================
# CONFIGURATION: Set your specific file paths
# ==========================================

# Your Himawari L3 TIFF file over Vietnam
HIMAWARI_PATH = "/home/slow_data/Air_Quality/AOD/L3_AOD/"

# Original MODIS MCD19A2 HDF4 file
MODIS_PATH = '/home/slow_data/Air_Quality/MODIS_MCD19A2'

# Output file path
OUTPUT_PATH = "/home/slow_data/Air_Quality/Merged_AOD"

In [ ]:
# Find all MCD19A2 HDF files
# We use path join with wildcards: MODIS_PATH / YYYY / day_in_year / filename
# pattern: /home/slow_data/Air_Quality/MODIS_MCD19A2/*/*/MCD19A2*.hdf
search_pattern = os.path.join(MODIS_PATH, '*', '*', 'MCD19A2*.hdf')
modis_files = glob.glob(search_pattern)

# Sort files to ensure order (optional but helpful for display)
modis_files.sort()

print(f"Found {len(modis_files)} HDF files in {MODIS_PATH}")

if len(modis_files) > 0:
    print("\nFirst 5 files:")
    for f in modis_files[:5]:
        # Display relative path to show the structure (Year/Day/File)
        rel_path = os.path.relpath(f, MODIS_PATH)
        print(f"  {rel_path}")
else:
    print(f"\nNo files found matching pattern: {search_pattern}")
    print("Please check if the directory structure matches: BASE_DIR/YYYY/DDD/filename.hdf")